In [ ]:
import os
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=4"
import jax
import jax_rmhd as jr
import jax.numpy as jnp
import jax.numpy.fft as ft
import matplotlib.pyplot as plt
jr.init_cluster()

#parameters
nx = 128
ny = 128
nz = 128
Lx = 2.0 * jnp.pi
Ly = 2.0 * jnp.pi
Lz = 2.0 * jnp.pi
t = 0.0
nsnap = 100
t_snap = 100
t_end = 10.0
cfl_safety = 0.5 
spatial_dimensions=3
snap_path="data/engine_test_sharding"

#we will use hyperviscosity
visc=1e-9
res=1e-9
hyper=3

mngr=jr.snapshot_manager_setup(snap_path=snap_path,nsnap=nsnap)

#prepare necessary objects for simulation
params=jr.Parameters(nx=nx,ny=ny,nz=nz,Lx=Lx,Ly=Ly,Lz=Lz,diss=(visc,res),hyper=hyper,cfl_safety=cfl_safety,dims=spatial_dimensions)
kgrid = jr.setup_kgrids(params)

def test_init(x,y,z):
    phi = (jnp.cos(x+1.4) + jnp.cos(y+0.5)) * jnp.sin(z)
    psi = (jnp.cos(2.0*x+2.3) + jnp.cos(y + 4.1)) * jnp.cos(z)
    return jnp.stack([phi,psi],axis=0)

state=jr.initialize(test_init,params)

nblock = jr.estimate_good_nblock(state,kgrid,params,t_snap,t_end,nblock_min=1)
print("nblock estimate: "+str(nblock)) #not actually using this, since we just want to simulate for a fixed 100 steps
nblock = 1

end_state=jr.simulate_scan(state,kgrid,params,nblock,t_snap=t_snap,t_end=t_end,mngr=mngr,save=False)

rmhd-solver has initialized jax in 64bit precision.
Running in local mode. Total devices: 1


nblock estimate: 8


In [ ]:
import jax_rmhd.snapshot_io as sn
#Making some snapshots.
for isnap in range(0,49):
    snap=sn.load_snapshot(isnap,mngr,params)
    #vort=ft.irfft2(-kgrid.ksq()*snap.fields.phik)
    #vsq=jnp.sum(jnp.array(jax.tree_util.tree_map(lambda gfk: ft.irfft2(gfk),gradk(snap.fields.phik,kgrid)))**2.0,axis=0)
    vort=ft.irfft2(-kgrid.ksq()*snap.fields.phik)
    plt.subplot(121)
    plt.imshow(vort[64,:,:],vmin=-30,vmax=30,cmap="afmhot")
    plt.subplot(122)
    plt.imshow(vort[:,0,:],vmin=-30,vmax=30,cmap="afmhot")
    plt.savefig(snap_path+str(isnap).zfill(3)+".png")
    plt.close('all')
    

In [ ]:
isnap=48
snap=sn.load_snapshot(isnap,mngr,params)
vort=ft.irfft2(-kgrid.ksq()*snap.fields.phik)
for iz in range(nz):
    plt.imshow(vort[iz,:,:],cmap="afmhot")
    plt.savefig(snap_path+"slice"+str(iz).zfill(3)+".png")
    plt.close('all')